Project repository: [josepsuy/LPR-GNN](https://github.com/josepsuy/LPR-GNN) (currently private; owner-granted access required). See [README.md](README.md) for installation and [CITATION.cff](CITATION.cff) for the preferred scientific citation.

# RCPSP directory makespan predictor

This notebook recursively evaluates `.rcp` and `.sm` instances from a directory using the trained LPR-GNN checkpoint and complete SSGS rollouts. CPU workers prepare instances and dynamic graphs; batched inference shares one model on the configured device. Results are periodically saved to Excel, and previously processed instances can be skipped when resuming.

A state with one Border activity is resolved directly, without constructing a graph or calling the neural model. If a batch fails, its instances are retried separately so that independent results can still be saved.

Raw instances are not bundled. Before running this notebook, follow [DATASETS.md](DATASETS.md) for download sources, placement and frozen-split requirements.


## 1. Project directory

In [ ]:
from pathlib import Path
import sys


PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    parent = PROJECT_ROOT.parent
    if (parent / "src").is_dir():
        PROJECT_ROOT = parent
    else:
        raise FileNotFoundError(
            "Cannot find src. Open the notebook from the project root."
        )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").is_dir())
print("config.yaml exists:", (PROJECT_ROOT / "config.yaml").is_file())


## 2. Imports and device


In [ ]:
from __future__ import annotations

import json
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from src.checkpoints import load_checkpoint, load_model_state
from src.rcpsp_benchmark import (
    get_bound,
    load_bounds_file,
    normalize_instance_name,
)
from src.normalization import GraphNormalizer
from src.rcpsp_instance import parse_mrcpsp
from src.rcpsp_schedule import calculate_makespan, calculate_makespans_batched
from src.trainer import build_model, load_config, resolve_device

print("PyTorch:", torch.__version__)
try:
    import torch_geometric
    print("PyTorch Geometric:", torch_geometric.__version__)
except Exception as exc:
    print("Could not import PyTorch Geometric:", exc)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print("VRAM total (GB):", round(total_vram, 2))

## 3. Model configuration


In [ ]:
CONFIG_PATH = PROJECT_ROOT / "config.yaml"
CHECKPOINT_PATH = PROJECT_ROOT / load_config(CONFIG_PATH)["output_dir"] / "best.pt"


## 4. Load the checkpoint

In [ ]:
config = load_config(CONFIG_PATH)
device = resolve_device(str(config.get("device", "auto")))
model = build_model(config).to(device)

checkpoint = load_checkpoint(CHECKPOINT_PATH, device)
load_model_state(model, checkpoint)
model.eval()
normalization_cfg = config.get("normalization", {})
graph_normalizer = GraphNormalizer(
    temporal_reference=float(normalization_cfg.get("temporal_reference", 100.0))
)

print("Device:", device)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Dynamic normalization: enabled")
if isinstance(checkpoint, dict):
    print("Epoch checkpoint:", checkpoint.get("epoch"))
    print("Best validation mean makespan:", checkpoint.get("best_mean_makespan"))

## 5. Batched directory benchmark

The recursive search includes `.rcp` and `.sm` files. The configured CPU workers load instances and build graphs; the GPU receives one batch per decision step. Only the main thread updates Excel. `SAVE_EVERY` controls how many new results accumulate before saving, avoiding the quadratic cost of rewriting the workbook after every file.


In [ ]:
INSTANCE_DIR = PROJECT_ROOT / "rcpsp" 
BOUNDS_PATH = None  # Optional relative reference file; no bounds are bundled.
RESULTS_XLSX = PROJECT_ROOT / config["output_dir"] / "makespan_results.xlsx"
MAX_INSTANCES = 0  # 0 = all instances
SKIP_ALREADY_PROCESSED = True
RECURSIVE = True  # include all subdirectories of INSTANCE_DIR

inference_cfg = config.get("inference", {})
ROLLOUT_BATCH_SIZE = int(inference_cfg.get("rollout_batch_size", 12))
N_WORKERS = int(inference_cfg.get("graph_workers", 12))
SAVE_EVERY = int(inference_cfg.get("save_every", 48))

print("INSTANCE_DIR:", INSTANCE_DIR)
print("BOUNDS_PATH:", BOUNDS_PATH)
print("RESULTS_XLSX:", RESULTS_XLSX)
print("RECURSIVE:", RECURSIVE)
print("ROLLOUT_BATCH_SIZE:", ROLLOUT_BATCH_SIZE)
print("N_WORKERS:", N_WORKERS)
print("SAVE_EVERY:", SAVE_EVERY)

In [ ]:
def suggested_horizon(instance, known_bound=None):
    serial_upper_bound = sum(int(values[0]) for values in instance.durations) + 1
    if known_bound is not None and int(known_bound) > 0:
        return max(serial_upper_bound, 2 * int(known_bound))
    return serial_upper_bound


def save_results_excel(dataframe, path):
    path = Path(path).with_suffix(".xlsx")
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(f".{path.stem}.tmp{path.suffix}")
    frame = dataframe.copy()
    if "elapsed_seconds" in frame:
        frame["elapsed_seconds"] = pd.to_numeric(
            frame["elapsed_seconds"], errors="coerce"
        ).round(5)
    try:
        with pd.ExcelWriter(temporary_path, engine="openpyxl") as writer:
            frame.to_excel(writer, index=False, sheet_name="results")
            worksheet = writer.book["results"]
            for column_cells in worksheet.columns:
                width = max(
                    len(str(cell.value)) if cell.value is not None else 0
                    for cell in column_cells
                )
                worksheet.column_dimensions[column_cells[0].column_letter].width = min(
                    width + 2, 80
                )
        os.replace(temporary_path, path)
    except Exception:
        if temporary_path.exists():
            temporary_path.unlink()
        raise
    return path


def merge_results(base_frame, new_rows):
    if not new_rows:
        return base_frame.copy()
    combined = pd.concat([base_frame, pd.DataFrame(new_rows)], ignore_index=True)
    combined["_file_key"] = combined["file"].astype(str).map(normalize_instance_name)
    combined = (
        combined.drop_duplicates(subset="_file_key", keep="last")
        .sort_values("_file_key", kind="stable")
        .drop(columns="_file_key")
        .reset_index(drop=True)
    )
    return combined


def prepare_instance(instance_path):
    bound = get_bound(instance_path.name, bounds, default=-1)
    instance = parse_mrcpsp(instance_path)
    horizon = suggested_horizon(instance, bound if bound > 0 else None)
    return instance_path, bound, instance, horizon


def result_row(prepared, result, elapsed_seconds, batch_size, execution_mode):
    instance_path, bound, _, _ = prepared
    gap_abs = result.makespan - bound if bound > 0 else None
    gap_ratio = 100.0 * gap_abs / bound if bound > 0 else None
    return {
        "file": instance_path.name,
        "makespan_bound": bound,
        "makespan_gnn": result.makespan,
        "gap_abs": gap_abs,
        "gap_ratio": gap_ratio,
        "bound_found": bound > 0,
        # Effective amortized time: batch wall time / instances in the batch.
        "elapsed_seconds": round(elapsed_seconds, 5),
        "inference_batch_size": int(batch_size),
        "execution_mode": execution_mode,
        "solution": json.dumps(result.solution),
    }


def prepare_batch(instance_paths):
    prepared = []
    preparation_errors = []
    worker_count = min(N_WORKERS, len(instance_paths))
    with ThreadPoolExecutor(max_workers=worker_count) as executor:
        futures = {
            executor.submit(prepare_instance, path): path
            for path in instance_paths
        }
        for future in as_completed(futures):
            path = futures[future]
            try:
                prepared.append(future.result())
            except Exception as exc:
                preparation_errors.append({"file": path.name, "error": repr(exc)})
    order = {path: index for index, path in enumerate(instance_paths)}
    prepared.sort(key=lambda item: order[item[0]])
    return prepared, preparation_errors


def evaluate_prepared_singly(prepared_items, reason):
    single_rows = []
    single_errors = []
    for prepared in prepared_items:
        instance_path, _, instance, horizon = prepared
        started = time.perf_counter()
        try:
            result = calculate_makespan(
                model=model,
                instance=instance,
                max_time=horizon,
                device=device,
                graph_transform=graph_normalizer,
            )
        except Exception as exc:
            single_errors.append({
                "file": instance_path.name,
                "error": repr(exc),
                "batch_error": repr(reason),
            })
            continue
        single_rows.append(
            result_row(
                prepared,
                result,
                time.perf_counter() - started,
                batch_size=1,
                execution_mode="single_fallback",
            )
        )
    return single_rows, single_errors


def evaluate_prepared_batch(prepared_items):
    if not prepared_items:
        return [], []
    instances = [item[2] for item in prepared_items]
    horizons = [item[3] for item in prepared_items]
    started = time.perf_counter()
    try:
        results = calculate_makespans_batched(
            model=model,
            instances=instances,
            max_times=horizons,
            device=device,
            graph_transform=graph_normalizer,
            graph_workers=N_WORKERS,
            include_scores=False,
        )
    except Exception as exc:
        if torch.cuda.is_available() and "out of memory" in str(exc).lower():
            torch.cuda.empty_cache()
        print(
            f"WARNING: a batch failed with {len(prepared_items)}; "
            "retrying each instance individually.",
            repr(exc),
        )
        return evaluate_prepared_singly(prepared_items, exc)

    batch_seconds = time.perf_counter() - started
    effective_seconds = batch_seconds / len(prepared_items)
    rows = [
        result_row(
            prepared,
            result,
            effective_seconds,
            batch_size=len(prepared_items),
            execution_mode="batched",
        )
        for prepared, result in zip(prepared_items, results)
    ]
    return rows, []


if N_WORKERS < 1:
    raise ValueError("N_WORKERS must be at least 1.")
if ROLLOUT_BATCH_SIZE < 1:
    raise ValueError("ROLLOUT_BATCH_SIZE must be at least 1.")
if SAVE_EVERY < 0:
    raise ValueError("SAVE_EVERY must not be negative.")

if INSTANCE_DIR.is_dir():
    file_iterator = INSTANCE_DIR.rglob("*") if RECURSIVE else INSTANCE_DIR.iterdir()
    instance_files = sorted(
        path for path in file_iterator
        if path.is_file() and path.suffix.lower() in {".rcp", ".sm"}
    )
else:
    instance_files = []

bounds = load_bounds_file(BOUNDS_PATH) if BOUNDS_PATH is not None and BOUNDS_PATH.is_file() else {}

if RESULTS_XLSX.exists():
    benchmark_df = pd.read_excel(RESULTS_XLSX)
else:
    benchmark_df = pd.DataFrame(columns=[
        "file", "makespan_bound", "makespan_gnn", "gap_abs", "gap_ratio",
        "bound_found", "elapsed_seconds", "inference_batch_size",
        "execution_mode", "solution",
    ])

processed = set(
    benchmark_df["file"].astype(str).map(normalize_instance_name)
) if len(benchmark_df) else set()

pending_files = [
    path for path in instance_files
    if not (
        SKIP_ALREADY_PROCESSED
        and normalize_instance_name(path.name) in processed
    )
]
if MAX_INSTANCES:
    pending_files = pending_files[:MAX_INSTANCES]

rows = []
errors = []
unsaved_rows = 0
if pending_files:
    print(
        f"Processing {len(pending_files)} instances in batches of "
        f"up to {ROLLOUT_BATCH_SIZE}, with {N_WORKERS} CPU workers and {device}."
    )
    total_started = time.perf_counter()
    for batch_start in range(0, len(pending_files), ROLLOUT_BATCH_SIZE):
        batch_paths = pending_files[batch_start : batch_start + ROLLOUT_BATCH_SIZE]
        batch_number = batch_start // ROLLOUT_BATCH_SIZE + 1
        batch_count = (len(pending_files) + ROLLOUT_BATCH_SIZE - 1) // ROLLOUT_BATCH_SIZE
        batch_started = time.perf_counter()

        prepared, batch_errors = prepare_batch(batch_paths)
        batch_rows, rollout_errors = evaluate_prepared_batch(prepared)
        rows.extend(batch_rows)
        errors.extend(batch_errors)
        errors.extend(rollout_errors)
        unsaved_rows += len(batch_rows)

        completed = min(batch_start + len(batch_paths), len(pending_files))
        batch_wall = time.perf_counter() - batch_started
        print(
            f"Batch {batch_number}/{batch_count}: {completed}/{len(pending_files)} "
            f"files, successful={len(batch_rows)}, errors={len(batch_errors) + len(rollout_errors)}, "
            f"wall={batch_wall:.2f}s"
        )

        if SAVE_EVERY and unsaved_rows >= SAVE_EVERY:
            benchmark_df_current = merge_results(benchmark_df, rows)
            save_results_excel(benchmark_df_current, RESULTS_XLSX)
            unsaved_rows = 0

    benchmark_df = merge_results(benchmark_df, rows)
    if rows:
        saved_path = save_results_excel(benchmark_df, RESULTS_XLSX)
        total_wall = time.perf_counter() - total_started
        throughput = len(rows) / total_wall if total_wall > 0 else float("nan")
        print("Results saved to:", saved_path)
        print(f"Wall total: {total_wall:.2f}s; throughput: {throughput:.4f} instances/s")
    if errors:
        print(f"{len(errors)} instances failed; rerun this cell to retry them.")
        display(pd.DataFrame(errors))
else:
    print("No pending instances. Check INSTANCE_DIR or SKIP_ALREADY_PROCESSED.")

benchmark_df.tail()

## 6. Benchmark summary

In [ ]:
valid_df = benchmark_df[pd.to_numeric(benchmark_df["makespan_bound"], errors="coerce") > 0].copy()


if len(valid_df) > 0:
    valid_df["gap_abs"] = valid_df["makespan_gnn"] - valid_df["makespan_bound"]
    valid_df["gap_ratio"] = 100.0 * valid_df["gap_abs"] / valid_df["makespan_bound"]
    summary = pd.DataFrame([{
        "instances": len(valid_df),
        "reference_matches": int((valid_df["gap_abs"] == 0).sum()),
        "makespan_gnn_sum": valid_df["makespan_gnn"].sum(),
        "gap_abs_mean": valid_df["gap_abs"].mean(),
        "gap_abs_median": valid_df["gap_abs"].median(),
        "gap_ratio_mean": valid_df["gap_ratio"].mean(),
        "gap_ratio_median": valid_df["gap_ratio"].median(),
        "elapsed_mean": valid_df["elapsed_seconds"].mean(),
    }])
    display(summary)
else:
    print("No rows have a valid bound.")

## 7. Cumulative makespan

In [ ]:
if len(valid_df):
    plot_df = valid_df.reset_index(drop=True)
    plt.figure(figsize=(10, 6))
    plt.plot(plot_df["makespan_bound"].cumsum(), label="Cumulative bound")
    plt.plot(plot_df["makespan_gnn"].cumsum(), label="Cumulative GNN")
    plt.xlabel("Instance")
    plt.ylabel("Cumulative makespan")
    plt.title("Cumulative comparison")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()